In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import classification_report, confusion_matrix
from glob import glob
import random
import json




In [ ]:
import torch
import torch.nn as nn

class SeparableConv3d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, padding=1):
        super().__init__()
        self.depthwise = nn.Conv3d(in_channels, in_channels, kernel_size=kernel_size, padding=padding, groups=in_channels)
        self.pointwise = nn.Conv3d(in_channels, out_channels, kernel_size=1)
        self.bn = nn.BatchNorm3d(out_channels)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        return self.relu(self.bn(x))

class Inception3DModule(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.branch1 = nn.Conv3d(in_channels, 32, kernel_size=1)

        self.branch2 = nn.Sequential(
            nn.Conv3d(in_channels, 32, kernel_size=1),
            nn.Conv3d(32, 48, kernel_size=3, padding=1)
        )

        self.branch3 = nn.Sequential(
            nn.Conv3d(in_channels, 8, kernel_size=1),
            nn.Conv3d(8, 16, kernel_size=5, padding=2)
        )

        self.branch4 = nn.Sequential(
            nn.MaxPool3d(kernel_size=3, stride=1, padding=1),
            nn.Conv3d(in_channels, 32, kernel_size=1)
        )

    def forward(self, x):
        x1 = self.branch1(x)
        x2 = self.branch2(x)
        x3 = self.branch3(x)
        x4 = self.branch4(x)
        return torch.cat([x1, x2, x3, x4], dim=1)  # Output: 32 + 48 + 16 + 32 = 128

class LiteInceptionNet3D(nn.Module):
    def __init__(self, num_classes=24):
        super().__init__()
        self.init_conv = nn.Sequential(
            nn.Conv3d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm3d(32),
            nn.ReLU()
        )

        self.inception = Inception3DModule(32)   # Output: 128 channels
        self.pool1 = nn.MaxPool3d(2)  # → 64³

        self.conv2 = SeparableConv3d(128, 64)
        self.pool2 = nn.MaxPool3d(2)  # → 32³

        self.conv3 = SeparableConv3d(64, 128)
        self.pool3 = nn.MaxPool3d(2)  # → 16³

        self.conv4 = SeparableConv3d(128, 256)
        self.global_pool = nn.AdaptiveAvgPool3d((1, 1, 1))

        self.classifier = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.init_conv(x)
        x = self.inception(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.pool2(x)

        x = self.conv3(x)
        x = self.pool3(x)

        x = self.conv4(x)
        x = self.global_pool(x)

        x = x.view(x.size(0), -1)
        return self.classifier(x)




In [ ]:

# ------------------------------------------
# 4. FeatureNet-like 3D CNN Model
# ------------------------------------------
class FeatureNet3D(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm3d(32),
            nn.ReLU(),
            nn.MaxPool3d(2),  # 64³

            nn.Conv3d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm3d(64),
            nn.ReLU(),
            nn.MaxPool3d(2),  # 32³

            nn.Conv3d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm3d(128),
            nn.ReLU(),
            nn.MaxPool3d(2),  # 16³

            nn.Conv3d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm3d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool3d((1, 1, 1))  # Global pooling
        )
        self.classifier = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.net(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class PreActBlock3D(nn.Module):
    """Pre-activation version of BasicBlock with LeakyReLU"""
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.bn1 = nn.BatchNorm3d(in_channels)
        self.conv1 = nn.Conv3d(in_channels, out_channels,
                              kernel_size=3, stride=stride,
                              padding=1, bias=False)
        self.bn2 = nn.BatchNorm3d(out_channels)
        self.conv2 = nn.Conv3d(out_channels, out_channels,
                              kernel_size=3, stride=1,
                              padding=1, bias=False)
        self.leaky_relu = nn.LeakyReLU(0.1, inplace=True)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv3d(in_channels, out_channels,
                         kernel_size=1, stride=stride,
                         bias=False)
            )

    def forward(self, x):
        # Pre-activation
        out = self.leaky_relu(self.bn1(x))
        shortcut = self.shortcut(out) if isinstance(self.shortcut, nn.Sequential) else x
        out = self.conv1(out)

        out = self.leaky_relu(self.bn2(out))
        out = self.conv2(out)

        return out + shortcut

class ResNet3D(nn.Module):
    def __init__(self, block, num_blocks, num_classes=24, initial_channels=64):
        super().__init__()
        self.in_channels = initial_channels

        # Initial layers
        self.init_conv = nn.Conv3d(1, initial_channels,
                                  kernel_size=7, stride=2,
                                  padding=3, bias=False)
        self.init_pool = nn.MaxPool3d(kernel_size=3, stride=2, padding=1)

        # Residual stages
        self.stage1 = self._make_stage(block, 64, num_blocks[0], stride=1)
        self.stage2 = self._make_stage(block, 128, num_blocks[1], stride=2)
        self.stage3 = self._make_stage(block, 256, num_blocks[2], stride=2)
        self.stage4 = self._make_stage(block, 512, num_blocks[3], stride=2)
        self.stage5 = self._make_stage(block, 512, num_blocks[4], stride=1)

        # Final layers with residual connection
        self.global_pool = nn.AdaptiveAvgPool3d((1, 1, 1))
        self.fc1 = nn.Linear(512, 128)
        self.fc2 = nn.Linear(128, num_classes)
        self.dropout = nn.Dropout(0.3)
        self.fc_shortcut = nn.Linear(512, num_classes)  # Residual for classifier

        # Initialize weights
        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='leaky_relu')
            elif isinstance(m, nn.BatchNorm3d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.constant_(m.bias, 0)

    def _make_stage(self, block, channels, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks - 1)
        blocks = []
        for stride in strides:
            blocks.append(block(self.in_channels, channels, stride))
            self.in_channels = channels
        return nn.Sequential(*blocks)

    def forward(self, x):
        # Initial processing
        x = self.init_conv(x)
        x = self.init_pool(x)

        # Residual stages
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)
        x = self.stage5(x)

        # Classification with residual
        x = self.global_pool(x)
        x = x.view(x.size(0), -1)

        # Main branch
        fc_out = self.fc1(x)
        fc_out = F.leaky_relu(fc_out, 0.1)
        fc_out = self.dropout(fc_out)
        fc_out = self.fc2(fc_out)

        # Shortcut branch
        shortcut = self.fc_shortcut(x)

        return fc_out + shortcut  # Residual connection
# ResNet26_3D_PREACT_ResidualMLP
def ResNet26_3D_PREACT_ResidualMLP(num_classes=24):
    """Custom 23-layer 3D ResNet with pre-activation blocks"""
    return ResNet3D(PreActBlock3D, [2, 2, 2, 2, 2], num_classes)

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
from glob import glob
import random
import json

# Assuming LiteInceptionNet3D and other necessary model components
# (SeparableConv3d, Inception3DModule) are defined in a previous cell or imported.
# If not, you would need to include their definitions here or ensure they are accessible.

class Feature_Predictor:
    def __init__(self, model_path, model_name, label_map_path=None, model_type = 1):
        self.model_path = model_path
        self.model_name = model_name
        self.full_model_path = os.path.join(self.model_path, f"{self.model_name}.pth")
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.label_map = self._load_label_map(label_map_path)
        self.model_type = model_type
        self.model = self._load_model()
        self.model.eval() # Set model to evaluation mode


    def _load_label_map(self, label_map_path):
        if label_map_path and os.path.exists(label_map_path):
            try:
                with open(label_map_path, 'r') as f:
                    return json.load(f)
            except json.JSONDecodeError:
                print(f"Error decoding JSON from label map file: {label_map_path}")
                # Fallback to default or handle error
                print("Using default label map.")
                return {str(i): i for i in range(24)} # Default label map
            except Exception as e:
                print(f"Error loading label map: {e}")
                print("Using default label map.")
                return {str(i): i for i in range(24)} # Default label map
        else:
            print("Warning: Label map not found or path not provided. Using default range.")
            return {str(i): i for i in range(24)} # Default label map

    def _load_model(self):
        # Instantiate your model. Ensure the architecture matches the saved model.
        # Assuming LiteInceptionNet3D is defined elsewhere and accessible
        if self.model_type == 1:
            model = LiteInceptionNet3D(num_classes=len(self.label_map)).to(self.device)
        elif self.model_type == 2:
            model = ResNet26_3D_PREACT_ResidualMLP(num_classes=len(self.label_map)).to(self.device)
        elif self.model_type == 3:
            model = FeatureNet3D(num_classes=len(self.label_map)).to(self.device)
        else:
            raise ValueError("Invalid model_type. Must be 1, 2, or 3.")

        try:
            checkpoint = torch.load(self.full_model_path, map_location=self.device)
            # Handle potential key mismatches if the model architecture changed slightly
            model.load_state_dict(checkpoint['model_state_dict'])
            print(f"Model loaded successfully from {self.full_model_path}")
        except FileNotFoundError:
            print(f"Error: Model file not found at {self.full_model_path}")
            # Handle the error appropriately, maybe raise an exception or exit
            raise FileNotFoundError(f"Model file not found at {self.full_model_path}")
        except RuntimeError as e:
            print(f"Error loading model state dictionary: {e}")
            print("This might be due to a mismatch between the model architecture and the saved state dict.")
            raise RuntimeError(f"Error loading model state dictionary: {e}")
        except Exception as e:
            print(f"An unexpected error occurred while loading the model: {e}")
            raise Exception(f"An unexpected error occurred while loading the model: {e}")

        return model

    def predict(self, data, threshold=0.01):
        """
        Performs prediction on the input data and returns top k predictions
        along with the top 1 prediction in different formats, including
        filtering by a probability threshold.

        Args:
            data (numpy.ndarray or torch.Tensor): Input data for prediction.
                                                 Expected format for numpy: (depth, height, width).
                                                 Expected format for torch.Tensor: (batch_size, channels, depth, height, width)
                                                 or (depth, height, width) if batch and channel dims are missing.
            threshold (float): Probability threshold for filtering top predictions.

        Returns:
            tuple: A tuple containing:
                - top1_label_str (str): The top 1 predicted class label as a string.
                - filtered_top_probs (dict): A dictionary containing class labels and
                                             probabilities for predictions above the threshold.
                - predicted_labels (numpy.ndarray): Predicted class labels for the top 1 prediction (original format).
        """
        if isinstance(data, np.ndarray):
            # Convert numpy to torch tensor and add batch and channel dimensions
            data = torch.tensor(data, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        elif isinstance(data, torch.Tensor):
            # Ensure the tensor has the correct number of dimensions
            if data.ndim == 3: # Assuming (depth, height, width)
                 data = data.unsqueeze(0).unsqueeze(0)
            elif data.ndim == 4: # Assuming (batch_size, depth, height, width)
                 data = data.unsqueeze(1) # Add channel dimension
            # If it's already 5D (batch_size, channels, depth, height, width), do nothing
        else:
            raise TypeError("Input data must be a numpy array or a torch tensor.")

        data = data.to(self.device)

        with torch.no_grad(): # Disable gradient calculation for inference
            outputs = self.model(data)
            probabilities = torch.softmax(outputs, dim=1) # Get probabilities
            _, predicted = torch.max(outputs.data, 1) # Top 1 prediction

        # Convert top 1 predicted class indices to labels using the label map
        idx_to_label = {v: k for k, v in self.label_map.items()}
        predicted_labels = np.array([idx_to_label[idx.item()] for idx in predicted])
        top1_label_str = predicted_labels[0] if predicted_labels.size > 0 else "N/A"

        # Filter probabilities based on the threshold and create a dictionary
        filtered_top_probs = {
            idx_to_label[i]: probabilities[0, i].item()
            for i in range(probabilities.size(1))
            if probabilities[0, i].item() > threshold
        }


        return top1_label_str, filtered_top_probs, predicted_labels
